# Dependencies

In [25]:
import pandas as pd

df = pd.read_csv('../data/raw/superstore.csv', encoding='latin1')

In [26]:
print(df.columns) # Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State' ...


Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')


## Normalizing Columns

In [27]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(' ', '_')
)

print(df.columns.tolist())

['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub-category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


# Exploration

In [28]:
print(f"How many rows and columns? -> {df.shape}\n")

print(f"What type is each column? \n\n{df.dtypes}")


How many rows and columns? -> (9994, 21)

What type is each column? 

row_id             int64
order_id             str
order_date           str
ship_date            str
ship_mode            str
customer_id          str
customer_name        str
segment              str
country              str
city                 str
state                str
postal_code        int64
region               str
product_id           str
category             str
sub-category         str
product_name         str
sales            float64
quantity           int64
discount         float64
profit           float64
dtype: object


In [29]:
print(f"What does the data look like? \n\n{df.head()}")

What does the data look like? 

   row_id        order_id  order_date   ship_date       ship_mode customer_id  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     customer_name    segment        country             city  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  postal_code  region       product_id         category sub-

In [30]:
print(f"Any missing values?\n\n{df.isnull().sum()}")

print(f"\n\nAny duplicate rows? -> {df.duplicated().sum()}")

Any missing values?

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub-category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64


Any duplicate rows? -> 0


# Phase 3 — Data Cleaning

In [31]:
## Data Types
print(df.dtypes)

row_id             int64
order_id             str
order_date           str
ship_date            str
ship_mode            str
customer_id          str
customer_name        str
segment              str
country              str
city                 str
state                str
postal_code        int64
region               str
product_id           str
category             str
sub-category         str
product_name         str
sales            float64
quantity           int64
discount         float64
profit           float64
dtype: object


In [32]:
# Date Str -> datetime
df['ship_date'] = pd.to_datetime(df['ship_date'])
df['order_date'] = pd.to_datetime(df['order_date'])

In [33]:
# Shipping time in days
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days

# Order year and month for time-series queries
df['order_year']  = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month

# Profit margin as a percentage
df['profit_margin'] = (df['profit'] / df['sales'] * 100).round(2)

# Flag unprofitable orders
df['is_unprofitable'] = df['profit'] < 0

# Verify Changes

In [34]:
print(df.dtypes)
print(df[['order_date', 'ship_date', 'days_to_ship', 'profit_margin']].head(10))

row_id                      int64
order_id                      str
order_date         datetime64[us]
ship_date          datetime64[us]
ship_mode                     str
customer_id                   str
customer_name                 str
segment                       str
country                       str
city                          str
state                         str
postal_code                 int64
region                        str
product_id                    str
category                      str
sub-category                  str
product_name                  str
sales                     float64
quantity                    int64
discount                  float64
profit                    float64
days_to_ship                int64
order_year                  int32
order_month                 int32
profit_margin             float64
is_unprofitable              bool
dtype: object
  order_date  ship_date  days_to_ship  profit_margin
0 2016-11-08 2016-11-11             3          16

## Save Cleaned Version

In [38]:
df.to_csv('../data/processed/superstore_clean.csv', index=False)

print(f"Saved {len(df)} rows to data/processed/ AS superstore_clean.csv")

Saved 9994 rows to data/processed/ AS superstore_clean.csv


# Load Data to PostgreSQL

In [67]:
df = pd.read_csv("../data/processed/superstore_clean.csv", encoding="latin1")

### Customers Table

In [68]:
customers_df = (
    df[['customer_id', 'customer_name', 'segment']]
    .drop_duplicates()
    .rename(columns={
        'customer_name': 'name'
    })
)

customers_df['customer_id'].duplicated().sum()

np.int64(0)

### Products Table

In [69]:
products_df = (
    df[['product_id', 'product_name', 'category', 'sub-category']]
    .drop_duplicates(subset=['product_id'])
    .rename(columns={
        'sub-category': 'sub_category'
    })
)

products_df['product_id'].duplicated().sum()

np.int64(0)

### Locations Table 

In [70]:
locations_df = (
    df[['city', 'state', 'region', 'postal_code']]
    .drop_duplicates()
    .reset_index(drop=True)
)

locations_df['location_id'] = locations_df.index + 1


df = df.merge(
    locations_df,
    on=['city', 'state', 'region', 'postal_code'],
    how='left'
)

locations_df['location_id'].duplicated().sum()

np.int64(0)

### Orders Table

In [73]:
orders_df = (
    df[['row_id','order_id', 'product_id', 'customer_id', 'location_id', 'order_date', 'ship_date', 'sales', 'quantity', 'discount', 'profit', 'profit_margin', 'days_to_ship', 'is_unprofitable']]
    .drop_duplicates()
)

orders_df['row_id'].duplicated().sum()

np.int64(0)

## Save Cleaned Tables

In [74]:
customers_df.to_csv("../data/processed/customers.csv", index=False)
products_df.to_csv("../data/processed/products.csv", index=False)
locations_df.to_csv("../data/processed/locations.csv", index=False)
orders_df.to_csv("../data/processed/orders.csv", index=False)

### PostgreSQL

In [75]:
from sqlalchemy import create_engine

DB_USER = "akmalyerzakov"
DB_PASSWORD = "$password$"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "superstore_db"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Tables
customers_df.to_sql(
    "customers",
    engine,
    if_exists="append",
    index=False
)

products_df.to_sql(
    "products",
    engine,
    if_exists="append",
    index=False
)

locations_df.to_sql(
    "locations",
    engine,
    if_exists="append",
    index=False
)

orders_df.to_sql(
    "orders",
    engine,
    if_exists="append",
    index=False
)

994